In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DateType,
    TimestampType
)
from datetime import datetime


# =========================
# Configuration
# =========================

SOURCE_BASE_PATH = (
    "abfss://raw@fintechdllasya.dfs.core.windows.net/"
    "transactions/2026-08-13/"
)

BRONZE_TABLE = (
    "dbx_fintech_data_platform.bronze.transactions"
)

INGESTION_LOG_TABLE = (
    "dbx_fintech_data_platform.metadata.ingestion_log"
)

PIPELINE_NAME = "transaction_bronze_ingestion"

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS dbx_fintech_data_platform.bronze
""")

spark.sql("""
CREATE SCHEMA IF NOT EXISTS dbx_fintech_data_platform.metadata
""")

In [0]:
log_schema = StructType([
    StructField("pipeline_name", StringType(), True),
    StructField("source_file", StringType(), True),
    StructField("source_date", DateType(), True),
    StructField("target_table", StringType(), True),
    StructField("status", StringType(), True),
    StructField("rows_processed", LongType(), True),
    StructField("started_at", TimestampType(), True),
    StructField("completed_at", TimestampType(), True),
    StructField("error_message", StringType(), True)
])

if not spark.catalog.tableExists(INGESTION_LOG_TABLE):
    (
        spark.createDataFrame([], log_schema)
        .write
        .format("delta")
        .saveAsTable(INGESTION_LOG_TABLE)
    )

In [0]:
source_files = [
    file.path
    for file in dbutils.fs.ls(SOURCE_BASE_PATH)
    if file.name.lower().endswith(".csv")
]

print("Files discovered:", len(source_files))

for file_path in source_files:
    print(file_path)

In [0]:
for source_path in sorted(source_files):

    started_at = datetime.now()

    # ---------------------------------
    # Check whether file was processed
    # ---------------------------------

    existing_count = (
        spark.table(INGESTION_LOG_TABLE)
        .filter(F.col("source_file") == source_path)
        .filter(F.col("status") == "SUCCESS")
        .count()
    )

    if existing_count > 0:

        print(
            f"SKIPPED: Source file already processed → "
            f"{source_path}"
        )

        continue


    print(
        f"PROCESSING: {source_path}"
    )


    try:

        # ---------------------------------
        # Read source file
        # ---------------------------------

        transaction_df = (
            spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .csv(source_path)
        )


        # ---------------------------------
        # Add Bronze metadata
        # ---------------------------------

        bronze_df = (
            transaction_df
            .withColumn(
                "_ingestion_timestamp",
                F.current_timestamp()
            )

            .withColumn(
                "_source_file",
                F.col("_metadata.file_path")
            )

            .withColumn(
                "_source_date",
                F.to_date(
                    F.regexp_extract(
                        F.col("_metadata.file_path"),
                        r"/transactions/(\d{4}-\d{2}-\d{2})/",
                        1
                    )
                )
            )
        )


        # ---------------------------------
        # Count records
        # ---------------------------------

        rows_processed = bronze_df.count()


        # ---------------------------------
        # Write to Bronze
        # ---------------------------------

        (
            bronze_df.write
            .format("delta")
            .mode("append")
            .saveAsTable(BRONZE_TABLE)
        )


        # ---------------------------------
        # Log successful ingestion
        # ---------------------------------

        completed_at = datetime.now()

        log_data = [(
            PIPELINE_NAME,
            source_path,
            datetime.strptime(
                source_path.split("/transactions/")[1][:10],
                "%Y-%m-%d"
            ).date(),
            BRONZE_TABLE,
            "SUCCESS",
            rows_processed,
            started_at,
            completed_at,
            None
        )]

        log_df = spark.createDataFrame(
            log_data,
            log_schema
        )

        (
            log_df.write
            .format("delta")
            .mode("append")
            .saveAsTable(INGESTION_LOG_TABLE)
        )


        print(
            f"SUCCESS: {source_path} | "
            f"Rows: {rows_processed}"
        )


    except Exception as e:

        # ---------------------------------
        # Log failed ingestion
        # ---------------------------------

        completed_at = datetime.now()

        error_log = [(
            PIPELINE_NAME,
            source_path,
            datetime.strptime(
                source_path.split("/transactions/")[1][:10],
                "%Y-%m-%d"
            ).date(),
            BRONZE_TABLE,
            "FAILED",
            0,
            started_at,
            completed_at,
            str(e)
        )]

        error_df = spark.createDataFrame(
            error_log,
            log_schema
        )

        (
            error_df.write
            .format("delta")
            .mode("append")
            .saveAsTable(INGESTION_LOG_TABLE)
        )

        print(
            f"FAILED: {source_path}"
        )

        print(
            f"Error: {str(e)}"
        )

In [0]:
bronze_transaction_df = spark.table(BRONZE_TABLE)

print(
    "Bronze transaction records:",
    bronze_transaction_df.count()
)

print(
    "Unique transactions:",
    bronze_transaction_df
    .select("transaction_id")
    .distinct()
    .count()
)

display(
    bronze_transaction_df
    .groupBy("_source_file")
    .count()
    .orderBy("_source_file")
)

display(
    bronze_transaction_df
    .groupBy("_source_date")
    .count()
)

display(
    spark.table(INGESTION_LOG_TABLE)
    .filter(
        F.col("pipeline_name") == PIPELINE_NAME
    )
    .orderBy("source_file")
)